In [1]:
import pandas as pd
import numpy as np
from math import log2

class Rule:
    def __init__(self, attributes, decision):
        self.conditions = attributes  # lista krotek (atrybut, wartość)
        self.decision = decision

def get_subtable(df, conditions):
    """Tworzy podtablicę na podstawie warunków."""
    sub_df = df.copy()
    for (attr, value) in conditions:
        sub_df = sub_df[sub_df[attr] == value]
    return sub_df

In [19]:
def heuristic_RM(current_df, decision_col='d'):
    attributes = current_df.columns.drop(decision_col)
    best_attr = None
    min_rm = float('inf')
    best_decision = None
    
    for attr in attributes:
        unique_values = current_df[attr].unique()
        for value in unique_values:
            sub_df = current_df[current_df[attr] == value]
            N_Tj1 = len(sub_df)
            N_Tj1_a = len(sub_df[sub_df[decision_col] == sub_df[decision_col].iloc[0]])
            M_Tj1_a = N_Tj1 - N_Tj1_a
            rm = M_Tj1_a / N_Tj1 
            # Poprawiony warunek: porównujemy nazwę atrybutu z best_attr[0]
            if (rm < min_rm or rm == min_rm) :
                min_rm = rm
                best_attr = (attr, value)
                if len(sub_df) > 0:
                    decision_counts = sub_df[decision_col].value_counts()
                    best_decision = decision_counts.index[0]  # Najczęstsza decyzja
                else:
                    best_decision = None    
                    
    return best_attr,10-min_rm,best_decision

In [3]:
def heuristic_POLY(current_df, decision_col='d'):
    attributes = current_df.columns.drop(decision_col)
    best_attr = None
    max_ratio = -float('inf')
    best_decision = None
    
    for attr in attributes:
        unique_values = current_df[attr].unique()
        for value in unique_values:
            sub_df = current_df[current_df[attr] == value]
            N_Tj = len(current_df)
            N_Tj_a = len(current_df[current_df[decision_col] == current_df[decision_col].iloc[0]])
            N_Tj1_a = len(sub_df[sub_df[decision_col] == sub_df[decision_col].iloc[0]])
            alpha = N_Tj_a - N_Tj1_a
            beta = (N_Tj - N_Tj_a) - (len(sub_df) - N_Tj1_a)
            ratio = beta / (alpha + 1) if (alpha + 1) != 0 else 0
            
            # Poprawiony warunek: porównujemy nazwę atrybutu z best_attr[0]
            if ratio > max_ratio or ratio == max_ratio :
                max_ratio = ratio
                best_attr = (attr, value)
                # Zapisz decyzję większościową dla tej podgrupy
                if len(sub_df) > 0:
                    decision_counts = sub_df[decision_col].value_counts()
                    best_decision = decision_counts.index[0]  # Najczęstsza decyzja
                else:
                    best_decision = None
    
    # Zwracamy tuple (best_attr, max_ratio, best_decision)
    return best_attr, max_ratio, best_decision

In [4]:
def induce_rule(df, heuristic, decision_col='d'):
    conditions = []
    used_attributes = set()  # Track which attributes are already used
    
    for value, subtable in df.items():
        current_df = subtable.copy()
        if len(current_df[decision_col].unique()) == 1:
            continue
        
        best_attr_val = heuristic(current_df, decision_col)
        if best_attr_val is None:
            continue
                
        attr, value, decision = best_attr_val
        
        conditions.append(((attr, value), decision, value))
    
    if not conditions:
        return Rule([], None)
    
    max_value = max(condition[2] for condition in conditions)
    
    # Wybierz tylko warunki z najwyższą wartością
    conditions_ret = []
    for condition in conditions:
        if condition[2] == max_value and condition[0][0]not in used_attributes:
                used_attributes.add(condition[0][0])
                new_entry = (condition[0], condition[2])
                conditions_ret.append(new_entry)
               
    return Rule(conditions_Fret, decision)

In [ ]:
from collections import Counter
def induce_rule_improved(df, heuristic, decision_col='d'):
    conditions = []
    used_attributes = set()  # Track which attributes are already used
    
    for value, subtable in df.items():
        current_df = subtable.copy()
        if len(current_df[decision_col].unique()) == 1:
            continue
        
        best_attr_val = heuristic(current_df, decision_col)
        if best_attr_val is None:
            continue
                
        attr, value, decision = best_attr_val
        
        conditions.append(((attr, value), decision, value))
    
    if not conditions:
        return Rule([], None)
    
    max_value = max(condition[2] for condition in conditions)
    
    # Wybierz warunki z najwyższą wartością
    conditions_with_max_value = []
    for condition in conditions:
        if condition[2] == max_value and condition[0][0][0] not in used_attributes:
            used_attributes.add(condition[0][0][0])
            new_entry = (condition[0], condition[2])
            conditions_with_max_value.append((new_entry, condition[1]))  # (condition, decision)
    
    # Znajdź najczęściej występującą decyzję w conditions_with_max_value
    if conditions_with_max_value:
        
        decisions_in_max_conditions = [item[1] for item in conditions_with_max_value]
        most_frequent_decision = Counter(decisions_in_max_conditions).most_common(1)[0][0]
        
        # Wybierz tylko te warunki które mają najczęstszą decyzję
        conditions_ret = [item[0] for item in conditions_with_max_value if item[1] == most_frequent_decision]
    else:
        conditions_ret = []
        most_frequent_decision = None        
    return Rule(conditions_ret, most_frequent_decision)

In [6]:
def split_dataframe_by_features(df, decision_column):
    feature_columns = df.columns.difference([decision_column])
    row_matches = {}
    
    for index, row in df.iterrows():
        row_matches[index] = {}
        
        for column in feature_columns:
            current_value = row[column]
            # Znajdź wszystkie wiersze z tą samą wartością w danej kolumnie
            matching_rows = df[df[column] == current_value].reset_index(drop=True)
            row_matches[index][column] = matching_rows
    
    return row_matches

In [7]:
def split_dataframe_by_features_optimized(df, decision_column):
   
    feature_columns = df.columns.difference([decision_column])
    
    # Pre-compute all grouped data once (major optimization)
    grouped_data = {}
    for column in feature_columns:
        # Store groups as reset_index DataFrames to match original output format
        grouped_data[column] = {
            value: group.reset_index(drop=True) 
            for value, group in df.groupby(column)
        }
    
    # Build result dictionary efficiently
    row_matches = {}
    for index, row in df.iterrows():
        row_matches[index] = {}
        for column in feature_columns:
            current_value = row[column]
            # Direct lookup instead of filtering - O(1) vs O(n)
            row_matches[index][column] = grouped_data[column].get(
                current_value, pd.DataFrame().reset_index(drop=True)
            )
    
    return row_matches

In [8]:
def rule_support_optimized(rule, table, decision_col='d'):
    """Optimized rule support calculation that matches the original behavior"""
    filtered = table.copy()
    
    for condition in rule.conditions:
        # Handle the complex nested tuple structure: ((attr, value), heur)
        if isinstance(condition, tuple) and len(condition) >= 2:
            # First element should be ((attr, value), heur) or just (attr, value)
            first_part = condition[0]
            
            if isinstance(first_part, tuple) and len(first_part) >= 2:
                # Check if first_part is ((attr, value), heur)
                if isinstance(first_part[0], tuple) and len(first_part[0]) >= 2:
                    attr = first_part[0][0]  # attribute name
                    val = first_part[0][1]   # attribute value
                else:
                    # first_part is (attr, value)
                    attr = first_part[0]
                    val = first_part[1]
                
                # Apply filter
                if attr in filtered.columns:
                    filtered = filtered[filtered[attr] == val]
                else:
                    return 0.0
            else:
                continue
        else:
            continue
    
    # Calculate support matching the original formula
    if len(filtered) == 0:
        return 0.0
        
    matching_decisions = (filtered[decision_col] == rule.decision).sum()
    # Use table.size like in original (total number of elements, not rows)
    support = matching_decisions / table.size
    
    return support

In [9]:
def rule_support(rule, table, decision_col='d'):
    # Filtruj tabelę według wszystkich warunków
    filtered = table
    for ar, heur in rule.conditions:
        attr = ar[0][0]
        val = ar[0][1]
        filtered = filtered[filtered[attr] == val]
    # Teraz policz ile tych rekordów ma odpowiednią decyzję
    support = (filtered[decision_col] == rule.decision).sum()/table.size
    return support

In [10]:
def generate_rule_summary(df, decision_attribute):
    results = []
    opis=0
    subtables = split_dataframe_by_features_optimized(df, decision_attribute)
    for feature, values_dict in subtables.items():
            # --- POLY Rule ---
        opis+=1
        rule_poly = induce_rule(values_dict, heuristic_POLY, 'd')
        
        if len(rule_poly.conditions) == 0:
            rule_poly_text = "Brak reguły decyzyjnej"
            support_poly = "-"
            length_poly = "-"
        else:
            condition_texts = []
            for condition in rule_poly.conditions:
                # condition ma format: ((('f1', 0), 1.0), 1.0)
                # Potrzebujemy wyciągnąć ('f1', 0) i użyć f1=0
                if isinstance(condition, tuple) and len(condition) >= 2:
                    first_part = condition[0]  # (('f1', 0), 1.0)
                    
                    if isinstance(first_part, tuple) and len(first_part) >= 2:
                        attr_value_pair = first_part[0]  # ('f1', 0)
                        
                        if isinstance(attr_value_pair, tuple) and len(attr_value_pair) >= 2:
                            attr_name = attr_value_pair[0]  # 'f1'
                            attr_value = attr_value_pair[1]  # 0
                            condition_texts.append(f'{attr_name}={attr_value}')
                        else:
                            condition_texts.append(str(condition))
                    else:
                        condition_texts.append(str(condition))
                else:
                    condition_texts.append(str(condition))
            
            condition_text_poly = ' AND '.join(condition_texts)
            rule_poly_text = f"IF {condition_text_poly} THEN d={rule_poly.decision}"
            
            try:
                support_poly = rule_support(rule_poly,df, 'd')
                length_poly = len(rule_poly.conditions)
            except Exception as e:
                support_poly = "Error"
                length_poly = len(rule_poly.conditions)
    
        rule_rm = induce_rule( values_dict, heuristic_RM, decision_attribute)
        if len(rule_rm.conditions) == 0:
            rule_rm_text = "Brak reguły decyzyjnej"
            support_rm = "-"
            length_rm = "-"
        else:
            condition_texts = []
            for condition in rule_rm.conditions:
                # condition ma format: ((('f1', 0), 1.0), 1.0)
                # Potrzebujemy wyciągnąć ('f1', 0) i użyć f1=0
                if isinstance(condition, tuple) and len(condition) >= 2:
                    first_part = condition[0]  # (('f1', 0), 1.0)
                    
                    if isinstance(first_part, tuple) and len(first_part) >= 2:
                        attr_value_pair = first_part[0]  # ('f1', 0)
                        
                        if isinstance(attr_value_pair, tuple) and len(attr_value_pair) >= 2:
                            attr_name = attr_value_pair[0]  # 'f1'
                            attr_value = attr_value_pair[1]  # 0
                            condition_texts.append(f'{attr_name}={attr_value}')
                        else:
                            condition_texts.append(str(condition))
                    else:
                        condition_texts.append(str(condition))
                else:
                    condition_texts.append(str(condition))
            
            condition_text_rm = ' AND '.join(condition_texts)
            rule_rm_text = f"IF {condition_text_rm} THEN d={rule_rm.decision}"
            
            try:
                support_rm = rule_support(rule_rm, df, 'd')
                length_rm = len(rule_rm.conditions)
            except Exception as e:
                support_rm = "Error"
                length_rm = len(rule_rm.conditions)

            # Append results
        results.append({
                'Opis': f'Wiersz {opis}',
                'Reguła RM': rule_rm_text,
                'Wsparcie RM': support_rm,
                'Długość RM': length_rm,
                'Reguła POLY': rule_poly_text,
                'Wsparcie POLY': support_poly,
                'Długość POLY': length_poly
            })

    return pd.DataFrame(results)

In [22]:
def parse_condition_text_optimized(conditions):
    """Extract condition text efficiently without repetitive parsing."""
    condition_texts = []
    for condition in conditions:
        if isinstance(condition, tuple) and len(condition) >= 2:
            first_part = condition[0]
            if isinstance(first_part, tuple) and len(first_part) >= 2:
                attr_value_pair = first_part[0]
                if isinstance(attr_value_pair, tuple) and len(attr_value_pair) >= 2:
                    attr_name = attr_value_pair[0]
                    attr_value = attr_value_pair[1]
                    condition_texts.append(f'{attr_name}={attr_value}')
                else:
                    condition_texts.append(str(condition))
            else:
                condition_texts.append(str(condition))
        else:
            condition_texts.append(str(condition))
    return condition_texts

def generate_rule_summary_optimized(df, decision_attribute):
    """Optimized version of generate_rule_summary with better performance."""
    results = []
    opis = 0
    
    # Use optimized split function
    subtables = split_dataframe_by_features_optimized(df, decision_attribute)
    
    for feature, values_dict in subtables.items():
        opis += 1
        
        # Generate rules for both heuristics
        rule_poly = induce_rule_improved(values_dict, heuristic_POLY, decision_attribute)
        rule_rm = induce_rule_improved(values_dict, heuristic_RM, decision_attribute)
        
        # Process POLY rule
        if len(rule_poly.conditions) == 0:
            rule_poly_text = "Brak reguły decyzyjnej"
            support_poly = "-"
            length_poly = "-"
        else:
            condition_texts = parse_condition_text_optimized(rule_poly.conditions)
            condition_text_poly = ' AND '.join(condition_texts)
            rule_poly_text = f"IF {condition_text_poly} THEN d={rule_poly.decision}"
            
            try:
                support_poly = rule_support_optimized(rule_poly, df, decision_attribute)
                length_poly = len(rule_poly.conditions)
            except Exception as e:
                support_poly = "Error"
                length_poly = len(rule_poly.conditions)
        
        # Process RM rule
        if len(rule_rm.conditions) == 0:
            rule_rm_text = "Brak reguły decyzyjnej"
            support_rm = "-"
            length_rm = "-"
        else:
            condition_texts = parse_condition_text_optimized(rule_rm.conditions)
            condition_text_rm = ' AND '.join(condition_texts)
            rule_rm_text = f"IF {condition_text_rm} THEN d={rule_rm.decision}"
            
            try:
                support_rm = rule_support_optimized(rule_rm, df, decision_attribute)
                length_rm = len(rule_rm.conditions)
            except Exception as e:
                support_rm = "Error"
                length_rm = len(rule_rm.conditions)

        # Append results
        results.append({
            'Opis': f'Wiersz {opis}',
            'Reguła RM': rule_rm_text,
            'Wsparcie RM': support_rm,
            'Długość RM': length_rm,
            'Reguła POLY': rule_poly_text,
            'Wsparcie POLY': support_poly,
            'Długość POLY': length_poly
        })

    return pd.DataFrame(results)

In [12]:
def optimize_by_length(df):
    optimized_rules = []

    for idx, row in df.iterrows():
        lengths = {}
        heuristics = ['RM', 'POLY']
        
        for heuristic in heuristics:
            length_val = row.get(f'Długość {heuristic}')
            if pd.isna(length_val) or length_val == '-':
                lengths[heuristic] = float('inf')
            else:
                lengths[heuristic] = int(length_val)

        min_length = min(lengths.values())

        # Wybierz pierwszą heurystykę z minimalną długością
        selected_heuristic = next(h for h, l in lengths.items() if l == min_length)
        selected_rule = row[f'Reguła {selected_heuristic}']

        optimized_rules.append({
            'Opis': row['Opis'],
            'Długość reguły': min_length if min_length != float('inf') else '-',
            'Wybrana heurystyka': selected_heuristic,
            'Wybrane reguły': selected_rule
        })

    return pd.DataFrame(optimized_rules)

In [13]:
def analyze_rules_lenght(df_sum: pd.DataFrame,df:pd.DataFrame) -> pd.DataFrame:
    # Convert rule length columns to numeric values
    
    df_op=optimize_by_length(df_sum)
    df_sum['Długość RM'] = pd.to_numeric(df_sum['Długość RM'], errors='coerce')
    df_sum['Długość POLY'] = pd.to_numeric(df_sum['Długość POLY'], errors='coerce')
    df_op['Długość reguły'] = pd.to_numeric(df_op['Długość reguły'], errors='coerce')
    
    # Calculate metrics
    min_rule_length_rm = df_sum['Długość RM'].min()
    min_rule_length_poly= df_sum['Długość POLY'].min()
    min_rule_length_op= df_op['Długość reguły'].min()
    
    num_rules_rm = df_sum['Reguła RM'].nunique()
    num_rules_poly = df_sum['Reguła POLY'].nunique()
    num_rules_op = df_op['Wybrane reguły'].nunique()
    
    avg_rule_length_rm = df_sum['Długość RM'].sum() / len(df_sum)
    avg_rule_length_poly = df_sum['Długość POLY'].sum() / len(df_sum)
    avg_rule_length_op = df_op['Długość reguły'].sum() / len(df_op)
    
    max_num_rules_rm = df_sum['Długość RM'].max()
    max_num_rules_poly =df_sum['Długość POLY'].max()
    max_num_rules_op =df_op['Długość reguły'].max()
    
    selected_rule_rm=df_sum[df_sum['Długość RM'] == min_rule_length_rm].iloc[0]
    selected_rule_poly=df_sum[df_sum['Długość POLY'] == min_rule_length_poly].iloc[0]
    selected_rule_op=df_op[df_op['Długość reguły'] == min_rule_length_op].iloc[0]
    
    
    # Create result table
    results = pd.DataFrame({
        'Liczba wierszy': [df.shape[0]],
        'Liczba atrybutów': [df.shape[1]],
        'Minimalna długość reguły RM': [min_rule_length_rm],
        'Średnia liczba reguł RM': [avg_rule_length_rm],
        'Maksymalna liczba reguł RM': [max_num_rules_rm],
        'Liczba unikalnych reguł RM': [num_rules_rm],
        'Wybrana reguła RM':[selected_rule_rm['Reguła RM']],
        'Minimalna długość reguły POLY': [min_rule_length_poly],
        'Średnia liczba reguł POLY': [avg_rule_length_poly],
        'Maksymalna liczba reguł POLY': [max_num_rules_poly],
        'Liczba unikalnych reguł POLY': [num_rules_poly],
        'Wybrana reguła POLY':[selected_rule_poly['Reguła POLY']],
        'Minimalna długość reguły Optymalizacja': [min_rule_length_op],
        'Średnia liczba reguł Optymalizacja': [avg_rule_length_op],
        'Maksymalna liczba reguł Optymalizacja': [max_num_rules_op],
        'Liczba unikalnych reguł Optymalizacja': [num_rules_op],
        'Wybrana reguła Optymalizacja':[selected_rule_op['Wybrane reguły']]
    })

    return results

In [14]:
def optimize_by_support(df):
    optimized_rules = []

    for idx, row in df.iterrows():
        supports = {}
        heuristics = ['RM', 'POLY']
        
        for heuristic in heuristics:
            support_val = row.get(f'Wsparcie {heuristic}')
            if pd.isna(support_val) or support_val == '-':
                supports[heuristic] = -float('inf')
            else:
                supports[heuristic] = float(support_val)

        max_support = max(supports.values())

        # Wybierz pierwszą heurystykę z maksymalnym wsparciem
        selected_heuristic = next(h for h, s in supports.items() if s == max_support)
        selected_rule = row[f'Reguła {selected_heuristic}']

        optimized_rules.append({
            'Opis': row['Opis'],
            'Wsparcie reguły': max_support if max_support != -float('inf') else '-',
            'Wybrana heurystyka': selected_heuristic,
            'Wybrane reguły': selected_rule
        })

    return pd.DataFrame(optimized_rules)

In [15]:
def analyze_rules_support(df_sum: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    # Konwersja kolumn ze wsparciem na wartości numeryczne
    
    df_op = optimize_by_support(df_sum)
    df_sum['Wsparcie RM'] = pd.to_numeric(df_sum['Wsparcie RM'], errors='coerce')
    df_sum['Wsparcie POLY'] = pd.to_numeric(df_sum['Wsparcie POLY'], errors='coerce')
    df_op['Wsparcie reguły'] = pd.to_numeric(df_op['Wsparcie reguły'], errors='coerce')
    
    # Obliczenie metryk
    max_support_rm = df_sum['Wsparcie RM'].max()
    max_support_poly = df_sum['Wsparcie POLY'].max()
    max_support_op = df_op['Wsparcie reguły'].max()
    
    num_rules_rm = df_sum['Reguła RM'].nunique()
    num_rules_poly = df_sum['Reguła POLY'].nunique()
    num_rules_op = df_op['Wybrane reguły'].nunique()
    
    avg_support_rm = df_sum['Wsparcie RM'].mean()
    avg_support_poly = df_sum['Wsparcie POLY'].mean()
    avg_support_op = df_op['Wsparcie reguły'].mean()
    
    min_support_rm = df_sum['Wsparcie RM'].min()
    min_support_poly = df_sum['Wsparcie POLY'].min()
    min_support_op = df_op['Wsparcie reguły'].min()
    
    # Poprawione odwołania do df_op i maksymalnego wsparcia
    selected_rule_rm = df_sum[df_sum['Wsparcie RM'] == max_support_rm].iloc[0]
    selected_rule_poly = df_sum[df_sum['Wsparcie POLY'] == max_support_poly].iloc[0]
    selected_rule_op = df_op[df_op['Wsparcie reguły'] == max_support_op].iloc[0]
    
    # Tworzenie tabeli wyników
    results = pd.DataFrame({
        'Liczba wierszy': [df.shape[0]],
        'Liczba atrybutów': [df.shape[1]],
        'Maksymalne wsparcie RM': [max_support_rm],
        'Średnie wsparcie RM': [avg_support_rm],
        'Minimalne wsparcie RM': [min_support_rm],
        'Liczba unikalnych reguł RM': [num_rules_rm],
        'Wybrana reguła RM': [selected_rule_rm['Reguła RM']],
        'Maksymalne wsparcie POLY': [max_support_poly],
        'Średnie wsparcie POLY': [avg_support_poly],
        'Minimalne wsparcie POLY': [min_support_poly],
        'Liczba unikalnych reguł POLY': [num_rules_poly],
        'Wybrana reguła POLY': [selected_rule_poly['Reguła POLY']],
        'Maksymalne wsparcie Optymalizacja': [max_support_op],
        'Średnie wsparcie Optymalizacja': [avg_support_op],
        'Minimalne wsparcie Optymalizacja': [min_support_op],
        'Liczba unikalnych reguł Optymalizacja': [num_rules_op],
        'Wybrana reguła Optymalizacja': [selected_rule_op['Wybrane reguły']]
    })

    return results

In [38]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from copy import deepcopy

class RuleBasedClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, rule_strings):
        self.rule_strings = rule_strings  # Przechowuj reguły jako stringi
        self.parsed_rules = []            # Parsowane reguły (warunki jako funkcje)
        self.default_class = None
        
    def fit(self, X, y):
        # Parsowanie reguł i konwersja warunków na funkcje
        self.parsed_rules = []
        for rule_str in self.rule_strings:
            parsed = self.parse_rule(rule_str)
            if parsed is None:
                continue
            conditions, target_value = parsed
            condition_func = self._conditions_to_function(conditions)
            self.parsed_rules.append((condition_func, target_value))
            
        self.default_class = y.mode()[0] if not y.empty else None
        return self
    
    def predict(self, X):
        return [self._predict_single(row) for _, row in X.iterrows()]
    
    def _predict_single(self, row):
        for condition, cls in self.parsed_rules:
            if condition(row):
                return cls
        return self.default_class
    
    @staticmethod
    def parse_rule(rule_str):
        if not rule_str or pd.isna(rule_str):
            return None
        
        try:
            condition_part, decision_part = rule_str.split(' THEN ')
        except ValueError:
            return None
            
        _, target_value = decision_part.split('=')
        target_value = target_value.strip()
        
        conditions = condition_part.replace('IF ', '').split(' AND ')
        return (conditions, target_value)
    
    def _conditions_to_function(self, conditions):
        # Tworzy funkcję sprawdzającą warunki
        def check_conditions(row):
            for cond in conditions:
                attr, val = [x.strip() for x in cond.split('=')]
                if row[attr] != val:
                    return False
            return True
        return check_conditions

    def __getstate__(self):
        # Pomijaj nie-serializowalne funkcje
        state = self.__dict__.copy()
        state['parsed_rules'] = []  # Usuń funkcje
        return state
    
    def __setstate__(self, state):
        # Odtwórz funkcje po deserializacji
        self.__dict__.update(state)
        self.fit(None, None)  # Ponowne parsowanie reguł

def evaluate_rules_model(summary_df, data_df, target_col, cv=10):
    # Ekstrakcja reguł
    rules = {
        'RM': summary_df.iloc[0]['Wybrana reguła RM'],
        'POLY': summary_df.iloc[0]['Wybrana reguła POLY'],
        'Optymalizacja': summary_df.iloc[0]['Wybrana reguła Optymalizacja']
    }
    
    # Przygotowanie danych
    X = data_df.drop(target_col, axis=1)
    y = data_df[target_col]
    
    # Strategia walidacji
    cv_strategy = StratifiedKFold(n_splits=min(10, y.nunique()), shuffle=True)
    
    # Zbieranie wyników
    results = []
    
    for rule_name, rule_str in rules.items():
        # Tworzymy model tylko z jedną regułą
        model = RuleBasedClassifier(rule_strings=[rule_str])
        
        # Obliczamy accuracy
        scores = cross_val_score(model, X, y, cv=cv_strategy, scoring='accuracy')
        
        results.append({
            'Reguła': rule_name,
            'Accuracy Średnia': np.nanmean(scores),
            'Accuracy Std': np.nanstd(scores)
        })
    
    # Tworzymy finalną tabelę
    df_results = pd.DataFrame(results)
    df_results = df_results[[
        'Reguła','Accuracy Średnia', 'Accuracy Std'
    ]]
    
    return df_results.round(4)


In [ ]:

df_ex=pd.read_csv('test.csv')
generate_rule_summary_optimized(df_ex,'d')

,Opis,Reguła RM,Wsparcie RM,Długość RM,Reguła POLY,Wsparcie POLY,Długość POLY
0,Wiersz 1,IF f2=1 THEN d=B,0.1250,1,IF f2=0 THEN d=A,0.125,1
1,Wiersz 2,IF f3=0 AND f2=1 THEN d=B,0.0625,2,IF f2=1 THEN d=B,0.125,1
2,Wiersz 3,IF f3=0 AND f2=1 THEN d=B,0.0625,2,IF f2=1 THEN d=B,0.125,1
3,Wiersz 4,IF f3=0 THEN d=B,0.0625,1,IF f2=1 THEN d=B,0.125,1


In [26]:
df_can=pd.read_csv('modified_breast-cancer.csv')
df_summary_can = generate_rule_summary_optimized(df_can,'class')


In [27]:
df_summary_can_op_lenght=optimize_by_length(df_summary_can)
df_summary_can_merge_lenght=pd.concat([df_summary_can,df_summary_can_op_lenght.drop(columns=["Opis"])],axis=1)

df_summary_can_op_supp=optimize_by_support(df_summary_can)
df_summary_can_merge_supp=pd.concat([df_summary_can,df_summary_can_op_supp.drop(columns=["Opis"])],axis=1)



df_summary_rule_can_lenght=analyze_rules_lenght(df_summary_can,df_can)

df_summary_rule_can_supp=analyze_rules_support(df_summary_can,df_can)


results_lenght = evaluate_rules_model(
        summary_df=df_summary_rule_can_lenght,
        data_df=df_can,
        target_col='class'
    )

df_lenght_full_table_can = pd.concat([df_summary_can_merge_lenght, df_summary_rule_can_lenght, results_lenght],axis=1)




results_supp = evaluate_rules_model(
        summary_df=df_summary_rule_can_supp,
        data_df=df_can,
        target_col='class'
    )
df_supp_full_table_can = pd.concat([df_summary_can_merge_supp, df_summary_rule_can_supp, results_supp],axis=1)

In [28]:

df_car=pd.read_csv('modified_cars.csv')
df_summary_car = generate_rule_summary_optimized(df_car,'class')

In [29]:
df_summary_car_op_lenght=optimize_by_length(df_summary_car)
df_summary_car_merge_lenght=pd.concat([df_summary_car,df_summary_car_op_lenght.drop(columns=["Opis"])],axis=1)

df_summary_car_op_supp=optimize_by_support(df_summary_car)
df_summary_car_merge_supp=pd.concat([df_summary_car,df_summary_car_op_supp.drop(columns=["Opis"])],axis=1)



df_summary_rule_car_lenght=analyze_rules_lenght(df_summary_car,df_car)

df_summary_rule_car_supp=analyze_rules_support(df_summary_car,df_car)


results_lenght_car = evaluate_rules_model(
        summary_df=df_summary_rule_car_lenght,
        data_df=df_car,
        target_col='class'
    )

df_lenght_full_table_car = pd.concat([df_summary_car_merge_lenght, df_summary_rule_car_lenght, results_lenght_car],axis=1)




results_supp_car = evaluate_rules_model(
        summary_df=df_summary_rule_car_supp,
        data_df=df_car,
        target_col='class'
    )
df_supp_full_table_car = pd.concat([df_summary_car_merge_supp, df_summary_rule_car_supp, results_supp_car],axis=1)

In [33]:

df_house=pd.read_csv('modified_house-votes.csv')
df_summary_house = generate_rule_summary_optimized(df_house,'class-name')

In [39]:

df_summary_house_op_lenght=optimize_by_length(df_summary_house)
df_summary_house_merge_lenght=pd.concat([df_summary_house,df_summary_house_op_lenght.drop(columns=["Opis"])],axis=1)

df_summary_house_op_supp=optimize_by_support(df_summary_house)
df_summary_house_merge_supp=pd.concat([df_summary_house,df_summary_house_op_supp.drop(columns=["Opis"])],axis=1)



df_summary_rule_house_lenght=analyze_rules_lenght(df_summary_house,df_house)

df_summary_rule_house_supp=analyze_rules_support(df_summary_house,df_house)


results_lenght_house = evaluate_rules_model(
        summary_df=df_summary_rule_house_lenght,
        data_df=df_house,
        target_col='class-name'
    )

df_lenght_full_table_house = pd.concat([df_summary_house_merge_lenght, df_summary_rule_house_lenght, results_lenght_house],axis=1)




results_supp_house = evaluate_rules_model(
        summary_df=df_summary_rule_house_supp,
        data_df=df_house,
        target_col='class-name'
    )
df_supp_full_table_house = pd.concat([df_summary_house_merge_supp, df_summary_rule_house_supp, results_supp_house],axis=1)